In [ ]:
# ==========================================
# NLGCL Kaggle 运行指南 (All-in-One)
# ==========================================
# 本 Notebook 用于在 Kaggle 环境中配置并运行 NLGCL 模型。
# 
# ## 前置条件
# 1. **GPU 环境**: 请确保 Notebook 设置中 Accelerator 选择 GPU (如 GPU P100 或 T4)。
# 2. **数据集**: 请上传本地 `dataset` 文件夹到 Kaggle Datasets (命名推荐 `nlgcl-dataset`)，并挂载到 Notebook 中。
#    - Kaggle 挂载路径通常为 `/kaggle/input/nlgcl-dataset`。
#
# ------------------------------------------
# STEP 1: 环境检测与代码准备
# ------------------------------------------
import os

# 定义你的 GitHub 仓库地址 (请替换为你的实际地址)
GITHUB_REPO_URL = "https://github.com/Jinfeng-Xu/NLGCL.git" 
REPO_NAME = "NLGCL" # 仓库名

if os.path.exists('/kaggle/working'):
    # 在 Kaggle 环境下
    print("Detected Kaggle environment.")
    
    # 克隆代码 (如果不存在)
    if not os.path.exists(REPO_NAME):
        cmd = f"git clone {GITHUB_REPO_URL} {REPO_NAME}"
        print(f"Executing: {cmd}")
        os.system(cmd)
    
    # 进入代码目录
    if os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
        print(f"Changed directory to {os.getcwd()}")
else:
    print("Not in Kaggle environment, assuming local run.")
    # 如果在本地，不需要 clone，直接运行即可

# ------------------------------------------
# STEP 2: 安装依赖
# ------------------------------------------
!pip install -r requirements.txt

# 安装 PyG 及其相关依赖 (根据 CUDA 版本自动选择)
import torch
try:
    print(f"PyTorch Version: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"CUDA Version: {torch.version.cuda}")
        torch_version = torch.__version__.split('+')[0]
        cuda_version = torch.version.cuda.replace('.', '')
        wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"
        
        print(f"Installing PyG binaries from: {wheel_url}")
        !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f $wheel_url
    else:
        print("CUDA not available, installing CPU PyG dependencies...")
        !pip install torch_scatter torch_sparse
except Exception as e:
    print("自动安装失败，尝试通用安装...", e)
    !pip install torch_scatter torch_sparse

# ------------------------------------------
# STEP 3: 数据集准备 (自动寻找与链接)
# ------------------------------------------
import shutil

# 自动寻找 Kaggle Input 中的 dataset 目录
def find_dataset_root(search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if 'yelp.inter' in files:
            # 找到了 yelp.inter，说明它的父目录是 yelp，爷目录是我们需要的 dataset 根目录?
            # 不一定，RecBole 需要结构: dataset/yelp/yelp.inter
            # 所以我们需要找到包含 'yelp' 文件夹的目录，且 'yelp' 文件夹内有 'yelp.inter'
            
            # Case 1: we found yelp.inter at .../dataset/yelp/yelp.inter -> return .../dataset
            if os.path.basename(root) == 'yelp':
                return os.path.dirname(root)
            
            # Case 2: maybe flattening? RecBole creates dataset/yelp/ based on config 'dataset': 'yelp'
            pass
    return None

if os.path.exists('/kaggle/working'):
    current_dataset_link = 'dataset'
    
    # 1. 寻找挂载的数据集
    found_path = find_dataset_root()
    if not found_path:
        # 尝试默认路径
        potential_paths = [
            "/kaggle/input/nlgcl-dataset",
            "/kaggle/input/nlgcl-dataset/dataset",
            "/kaggle/input/dataset"
        ]
        for p in potential_paths:
            if os.path.exists(os.path.join(p, 'yelp', 'yelp.inter')):
                found_path = p
                break
    
    if found_path:
        print(f"Found dataset at: {found_path}")
        
        # 2. 清理旧的 dataset 目录/链接
        if os.path.exists(current_dataset_link):
            if os.path.islink(current_dataset_link):
                os.unlink(current_dataset_link)
            else:
                shutil.rmtree(current_dataset_link)
        
        # 3. 创建链接
        # 注意: RecBole 写入过滤后的 dataset 时需要写权限，Kaggle input 是只读的。
        # 所以不能直接 link 整个 dataset 目录，最好是 link 内部文件，或者复制。
        # 但复制 700MB 比较慢。RecBole 的 saved/ 目录是单独配置的，
        # 关键是它是否会在 dataset 目录下生成预处理文件?
        # 是的，RecBole 默认会在 dataset/yelp/ 下生成 yelp.inter, yelp-dataset.pth 等。
        # 如果是只读挂载，生成 .pth 会失败！
        
        print("Copying dataset to working directory (required for write access to generate processed files)...")
        # 复制是更稳妥的方式，确保有写权限
        shutil.copytree(found_path, current_dataset_link)
        print("Dataset copied successfully.")
        
    else:
        print("Error: Could not find 'yelp/yelp.inter' in /kaggle/input. Please verify your dataset upload.")
        # 列出 input 目录帮助调试
        !find /kaggle/input -maxdepth 3
else:
    print("Local environment, skipping dataset setup.")

!ls -l dataset/yelp/

# ------------------------------------------
# STEP 4: 强制写入过滤配置 (覆盖 yelp.yaml)
# ------------------------------------------
# 为什么之前的运行没有过滤？
# 1. yelp.inter 本身是 700MB+ 的全量数据 (1.9M 用户)。
# 2. RecBole 需要通过配置文件在加载时进行过滤。
# 3. 此处强制写入配置，确保 RecBole 执行过滤 (保留至少15次交互的用户/物品)。

yelp_config_content = """
load_col:
  inter: [user_id, item_id, rating]
ITEM_ID_FIELD: item_id
RATING_FIELD: rating

# 核心过滤配置：过滤交互少于 15 次的用户和物品
# 这会将 1.9M 用户过滤到约 45k 用户
user_inter_num_interval: "[15,inf)"
item_inter_num_interval: "[15,inf)"

val_interval:
  rating: "[3,inf)"

warm_up_step: 40
"""

os.makedirs('properties', exist_ok=True)
with open('properties/yelp.yaml', 'w') as f:
    f.write(yelp_config_content)
print("Updated properties/yelp.yaml with filtering rules.")

# ------------------------------------------
# STEP 5: 运行训练
# ------------------------------------------
# 清理旧的缓存，强制重新处理数据
if os.path.exists("saved"):
    import glob
    for pth in glob.glob("saved/*.pth"):
        os.remove(pth)

print("Starting training with correct configuration...")
# 添加 --config_files 参数确保加载 properties/yelp.yaml
!python main.py --dataset=yelp